# Encoding TEI with LLMs

This notebook demonstrates how to use Large Language Models (LLMs) to assist in the encoding of texts in the Text Encoding Initiative (TEI) format. The TEI is a standard for representing texts in digital form, and LLMs can help automate and enhance the encoding process.


In [1]:
from pathlib import Path
import os
from openai import OpenAI
from google import genai
from tqdm import tqdm

from dotenv import load_dotenv
load_dotenv()

True

In [2]:

def build_user_prompt(text: str) -> str:
    return f"""TEXTE À ENCODER :
{text}

TEXTE ENCODE EN XML TEI :
"""


def openrouter(client, model_name, pre_prompt, text):
    user_prompt = build_user_prompt(text)
    # Gemma → pas de system message
    if "gemma" in model_name.lower() or "mistral" in model_name.lower():

        full_prompt = pre_prompt + "\n\n" + user_prompt

        response = client.chat.completions.create(
            model=model_name,
            temperature=0,
            messages=[
                {"role": "user", "content": full_prompt},
            ],
        )
    # Autres modèles → messages normaux
    else:
        response = client.chat.completions.create(
            model=model_name,
            temperature=0,
            messages=[
                {"role": "system", "content": pre_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )
    return response.choices[0].message.content.strip()


def googleai(client, model_name, pre_prompt, text):
    user_prompt = build_user_prompt(text)
    response = client.models.generate_content(
        model=model_name,
        contents=pre_prompt + "\n\n" + user_prompt
    )
    return response.text


def tei_encoding(pre_prompt, text, provider, client, model_name):
    try:
        if provider == "openrouter":
            return openrouter(client, model_name, pre_prompt, text)
        elif provider == "googleai":
            return googleai(client, model_name, pre_prompt, text)
        else:
            raise ValueError("Provider inconnu")

    except Exception as e:
        print(f"Erreur {provider}: {e}")
        return text

In [ ]:

## **** A CONFIGURER ****
provider = "openrouter" #"googleai" or "googleai"

#models = ["openai/gpt-5-mini", "google/gemma-3-27b-it"] # valeurs possible pour OpenRouter: "openai/gpt-5-mini", "google/gemma-3-27b-it", "google/gemma-3-12b-it", ...
models = ["openai/gpt-5.4-mini", "google/gemini-3.1-flash-lite-preview"]
#models = ["gemma-3-27b-it"] # valeurs possibles pour GoogleAI : "gemma-3-4b-it", "gemma-3-12b-it", "gemma-3-27b-it"

#ocr_path_name = "data/glm-ocr/txt/"
ocr_path_name = "data/input"
ocr_files = sorted(Path(ocr_path_name).glob("*.txt"))

if provider == "googleai":
    client = genai.Client(
        api_key=os.getenv("GOOGLE_API_KEY")
        )
elif provider == "openrouter":
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),
    )

prompts = sorted(Path("data/prompts/").glob("*.txt"))

for prompt in prompts:
    with open(prompt) as f:
        pre_prompt = f.read()

    print("* "+prompt.stem)

    for model_name in models:
        if provider == "openrouter":
            model_short = model_name.split("/")[1].split(":")[0]
        elif provider == "googleai":
            model_short = model_name

        print("** "+ model_short)
        # create folder with model name if it doesn't exist
        output_path = Path(f"data/output/{model_short}")
        output_path.mkdir(parents=True, exist_ok=True)

        output_path = Path(f"data/output/{model_short}/{prompt.stem}")
        output_path.mkdir(parents=True, exist_ok=True)

        for ocr_file in tqdm(ocr_files):
            ocr_text = ocr_file.read_text(encoding="utf-8", errors="replace")

            # if file already exists, skip
            output_file = output_path / f"{ocr_file.stem}.xml"
            if output_file.exists():
                continue

            corrected_text = tei_encoding(pre_prompt, ocr_text, provider, client, model_name)
            
            # remove ```xml at the beginning and ``` at the end if they exist
            if corrected_text.startswith("```xml"):
                corrected_text = corrected_text[len("```xml"):].strip()
            if corrected_text.endswith("```"):
                corrected_text = corrected_text[:-len("```")].strip()

            # Write corrected text to output file
            output_file.write_text(corrected_text, encoding="utf-8", errors="replace")
